## Spark Structured Streaming in Databricks

### Setup Paths (Volume-Based)

In [0]:
input_path = "/Volumes/ev_lab/bronze/raw_files/ev_sessions_stream/"
checkpoint_path = "/Volumes/ev_lab/bronze/raw_files/checkpoints/ev_sessions/"

dbutils.fs.mkdirs(input_path)
dbutils.fs.mkdirs(checkpoint_path)

### Define Schema for Streaming

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("session_id", StringType()),
    StructField("station_id", StringType()),
    StructField("charger_id", StringType()),
    StructField("start_time", TimestampType()),
    StructField("end_time", TimestampType()),
    StructField("energy_kwh", DoubleType()),
    StructField("charging_rate_kw", DoubleType()),
    StructField("temperature_c", DoubleType()),
    StructField("status", StringType())
])

### Read Input Path as Streaming Source

In [0]:
stream_df = spark.readStream \
    .format("csv") \
    .schema(schema) \
    .option("header", "true") \
    .load(input_path)

### Write to a Bronze Delta Table

In [0]:
query = stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .toTable("ev_lab.bronze.ev_charging_sessions_stream")

### Simulate Incoming Data (New File Arrives!)

In [0]:
data1 = """session_id,station_id,charger_id,start_time,end_time,energy_kwh,charging_rate_kw,temperature_c,status
S101,EV_IND_001,C-IND-001,2025-06-01 10:00:00,2025-06-01 11:00:00,40,40,35,Completed
S102,EV_IND_001,C-IND-002,2025-06-01 10:15:00,2025-06-01 11:10:00,38,35,34,Completed
"""

with open(f"{input_path}/batch1.csv", "w") as f:
    f.write(data1)

In [0]:
%sql

SELECT * FROM ev_lab.bronze.ev_charging_sessions_stream

### Simulate More Incoming Data (Next File Batch)

In [0]:
data2 = """session_id,station_id,charger_id,start_time,end_time,energy_kwh,charging_rate_kw,temperature_c,status
S103,EV_IND_001,C-IND-003,2025-06-01 11:30:00,2025-06-01 12:00:00,0,0,33,Fault
S104,EV_IND_001,C-IND-004,2025-06-01 12:00:00,2025-06-01 13:00:00,45,45,36,Completed
"""

with open(f"{input_path}/batch2.csv", "w") as f:
    f.write(data2)

In [0]:
%sql

SELECT * FROM ev_lab.bronze.ev_charging_sessions_stream

### Stop the Stream

query.stop()

In [0]:
query.stop()